In [ ]:
!pip -q install flask pyngrok

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os
from pyngrok import ngrok, conf

NGROK_AUTHTOKEN = "3AZA6IkBxvNF8hdmbGC0iccxYGe_4bQeFpbHE5szgEcpv3ncS"
PORT = 8000
CFG_PATH = "/tmp/ngrok_colab_fresh.yml"

# matikan proses ngrok yang mungkin masih hidup di runtime ini
try:
    ngrok.kill()
except:
    pass

# hapus config temp lama kalau ada
if os.path.exists(CFG_PATH):
    os.remove(CFG_PATH)

# pakai config BARU, bukan config default lama
pyngrok_config = conf.PyngrokConfig(
    config_path=CFG_PATH,
    auth_token=NGROK_AUTHTOKEN
)
conf.set_default(pyngrok_config)

# simpan token ke config baru ini
ngrok.set_auth_token(NGROK_AUTHTOKEN, pyngrok_config=pyngrok_config)

print("Fresh ngrok config ready:", CFG_PATH)

Fresh ngrok config ready: /tmp/ngrok_colab_fresh.yml


In [ ]:
import os
import json
import csv
from datetime import datetime
from flask import Flask, request, jsonify

# Folder utama
BASE_DIR = "/content/drive/MyDrive/Data_Monitoring_Motor"

# Subfolder baru
FOLDER_NAME = "ESP32_Multimon_Baru"
SAVE_DIR = os.path.join(BASE_DIR, FOLDER_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

RAW_JSONL_PATH = os.path.join(SAVE_DIR, "multimon_raw.jsonl")
CSV_PATH = os.path.join(SAVE_DIR, "multimon_samples.csv")

app = Flask(__name__)

def ensure_csv_header():
    if not os.path.exists(CSV_PATH):
        with open(CSV_PATH, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([
                "server_time","device","n","idx","ts_ms",
                "VRS","VST","VTR","VRN","VSN","VTN",
                "IR","IS","IT",
                "PR_kW","PS_kW","PT_kW",
                "SR_kVA","SS_kVA","ST_kVA",
                "QR_kVAr","QS_kVAr","QT_kVAr",
                "pfR","pfS","pfT",
                "fR","fS","fT",
                "vib_speed_mm_s","vib_disp_um","vib_acc","Tmotor_C"
            ])

ensure_csv_header()

@app.route("/", methods=["GET"])
def home():
    return jsonify({"ok": True, "message": "server running"}), 200

@app.route("/health", methods=["GET"])
def health():
    return jsonify({"ok": True}), 200

@app.route("/ingest", methods=["POST"])
def ingest():
    try:
        data = request.get_json(force=True)
        if not data:
            return jsonify({"ok": False, "error": "empty json"}), 400

        with open(RAW_JSONL_PATH, "a", encoding="utf-8") as f:
            f.write(json.dumps(data, ensure_ascii=False) + "\n")

        device = data.get("device", "UNKNOWN")
        n = data.get("n", 0)
        samples = data.get("samples", [])

        with open(CSV_PATH, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            server_time = datetime.now().strftime("%Y-%m-%d %H:%M:%S")

            for idx, s in enumerate(samples):
                writer.writerow([
                    server_time, device, n, idx, s.get("ts_ms",""),
                    s.get("VRS",""), s.get("VST",""), s.get("VTR",""),
                    s.get("VRN",""), s.get("VSN",""), s.get("VTN",""),
                    s.get("IR",""), s.get("IS",""), s.get("IT",""),
                    s.get("PR_kW",""), s.get("PS_kW",""), s.get("PT_kW",""),
                    s.get("SR_kVA",""), s.get("SS_kVA",""), s.get("ST_kVA",""),
                    s.get("QR_kVAr",""), s.get("QS_kVAr",""), s.get("QT_kVAr",""),
                    s.get("pfR",""), s.get("pfS",""), s.get("pfT",""),
                    s.get("fR",""), s.get("fS",""), s.get("fT",""),
                    s.get("vib_speed_mm_s",""),
                    s.get("vib_disp_mm",""),   # field lama dari ESP32, anggap sebagai um
                    s.get("vib_acc",""),
                    s.get("Tmotor_C","")
                ])

        return jsonify({
            "ok": True,
            "saved_samples": len(samples),
            "csv_path": CSV_PATH,
            "raw_path": RAW_JSONL_PATH
        }), 200

    except Exception as e:
        return jsonify({"ok": False, "error": str(e)}), 500

In [ ]:
import threading

def run_app():
    app.run(host="0.0.0.0", port=PORT, debug=False, use_reloader=False)

thread = threading.Thread(target=run_app, daemon=True)
thread.start()

print(f"Flask running on port {PORT}")

Flask running on port 8000


In [ ]:
from pyngrok import ngrok

public_tunnel = ngrok.connect(
    addr=PORT,
    proto="http",
    bind_tls=True,
    pyngrok_config=pyngrok_config
)

public_url = public_tunnel.public_url

print("PUBLIC URL :", public_url)
print("INGEST URL :", public_url + "/ingest")
print("HEALTH URL :", public_url + "/health")

 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:8000
 * Running on http://172.28.0.12:8000
INFO:werkzeug:Press CTRL+C to quit


PUBLIC URL : https://trena-nonenervating-remonstrantly.ngrok-free.dev
INGEST URL : https://trena-nonenervating-remonstrantly.ngrok-free.dev/ingest
HEALTH URL : https://trena-nonenervating-remonstrantly.ngrok-free.dev/health


In [ ]:
import requests

r = requests.get(public_url + "/health", timeout=20)
print("STATUS:", r.status_code)
print("BODY  :", r.text)

INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 07:21:30] "GET /health HTTP/1.1" 200 -


STATUS: 200
BODY  : {"ok":true}



In [ ]:
import requests
r = requests.get("https://trena-nonenervating-remonstrantly.ngrok-free.dev/health", timeout=20)
print(r.status_code)
print(r.text)

INFO:werkzeug:127.0.0.1 - - [24/Mar/2026 07:21:31] "GET /health HTTP/1.1" 200 -


200
{"ok":true}

